# ChatGPT API with Redis Server 

The following code uses `chat_history` as only key for only session as POC for using Redis to store chat history.

In [8]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
import redis

load_dotenv()

client = OpenAI()
redis_host = os.getenv("REDIS_HOST", "localhost")
redis_port = int(os.getenv("REDIS_PORT", 6379))
redis_username = os.getenv("REDIS_USERNAME")
redis_password = os.getenv("REDIS_PASSWORD")
redis_client = redis.Redis(
    host=redis_host,
    port=redis_port,
    username=redis_username,
    password=redis_password,
    decode_responses=True,
)

MODEL = "gpt-4o-mini"
SYSTEM_PROMPT = "You are a helpful assistant."
TOOLS = [
    {
        "type": "web_search"
    }
]
CONFIG = {
    "temperature": 0.1,
    "max_output_tokens": 1000,
}
CHAT_HISTORY_KEY = "chat_history"


def load_history():
    raw_items = redis_client.lrange(CHAT_HISTORY_KEY, 0, -1)
    return [json.loads(item) for item in raw_items] if raw_items else []


def save_message(role, content):
    redis_client.rpush(CHAT_HISTORY_KEY, json.dumps({"role": role, "content": content}))


history = load_history()
if history:
    print(f"Loaded {len(history)} messages from Redis.")
else:
    print("No prior history found in Redis.")

while True:
    USER_QUESTION = input("Enter your message ('exit' or 'quit'): ")

    if USER_QUESTION.lower() in ["exit", "quit"]:
        print("Exiting...")
        break

    print("User:", USER_QUESTION)
    history.append({
        "role": "user",
        "content": USER_QUESTION
    })
    save_message("user", USER_QUESTION)

    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        tools=TOOLS,
        input=history,
        **CONFIG
    )

    answer = response.output_text
    print("AI:", answer)
    history.append({
        "role": "assistant",
        "content": answer
    })
    save_message("assistant", answer)

Loaded 8 messages from Redis.
User: hi what is my name?
AI: Your name is Tom! What would you like to talk about?
User: What is machine learning in 100 words
AI: Machine learning is a subset of artificial intelligence that enables systems to learn from data and improve their performance over time without explicit programming. It involves algorithms that analyze patterns in data, allowing computers to make predictions or decisions based on new inputs. Machine learning can be categorized into supervised learning, where models are trained on labeled data; unsupervised learning, which identifies patterns in unlabeled data; and reinforcement learning, where agents learn through trial and error. Applications range from image and speech recognition to recommendation systems and autonomous vehicles, transforming industries by automating complex tasks and enhancing decision-making.
Exiting...


The following uses `chat_id` to identify session user and manage their chat history. This allows multiple users.

In [7]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json
import redis

load_dotenv()

client = OpenAI()
redis_url = os.getenv("REDIS_URL")
if redis_url:
    redis_client = redis.from_url(redis_url, decode_responses=True)
else:
    redis_host = os.getenv("REDIS_HOST", "localhost")
    redis_port = int(os.getenv("REDIS_PORT", 6379))
    redis_username = os.getenv("REDIS_USERNAME")
    redis_password = os.getenv("REDIS_PASSWORD")
    redis_client = redis.Redis(
        host=redis_host,
        port=redis_port,
        username=redis_username,
        password=redis_password,
        decode_responses=True,
    )

try:
    redis_client.ping()
    print("Connected to Redis.")
except Exception as exc:
    raise RuntimeError(f"Redis connection failed: {exc}") from exc

chat_id = input("Enter chat id for this chat: ").strip()
if not chat_id:
    raise ValueError("Chat id is required to separate chat histories.")

MODEL = "gpt-4o-mini"
SYSTEM_PROMPT = "You are a helpful assistant."
TOOLS = [
    {
        "type": "web_search"
    }
]
CONFIG = {
    "temperature": 0.1,
    "max_output_tokens": 1000,
}
CHAT_HISTORY_KEY = f"chat_history:{chat_id}"


def load_history():
    raw_items = redis_client.lrange(CHAT_HISTORY_KEY, 0, -1)
    return [json.loads(item) for item in raw_items] if raw_items else []


def save_message(role, content):
    redis_client.rpush(CHAT_HISTORY_KEY, json.dumps({"role": role, "content": content}))


history = load_history()
if history:
    print(f"Loaded {len(history)} messages from Redis.")
else:
    print("No prior history found in Redis.")

while True:
    USER_QUESTION = input("Enter your message ('exit' or 'quit'): ")

    if USER_QUESTION.lower() in ["exit", "quit"]:
        print("Exiting...")
        break

    print("User:", USER_QUESTION)
    history.append({
        "role": "user",
        "content": USER_QUESTION
    })
    save_message("user", USER_QUESTION)

    response = client.responses.create(
        model=MODEL,
        instructions=SYSTEM_PROMPT,
        tools=TOOLS,
        input=history,
        **CONFIG
    )

    answer = response.output_text
    print("AI:", answer)
    history.append({
        "role": "assistant",
        "content": answer
    })
    save_message("assistant", answer)

Connected to Redis.
Loaded 4 messages from Redis.
User: hi what is my name
AI: Your name is Thomas. How can I help you today?
User: describe Japan in 100 words
AI: Japan is an island nation in East Asia, known for its rich culture, advanced technology, and stunning landscapes. It blends ancient traditions, such as tea ceremonies and sumo wrestling, with modern innovations like robotics and high-speed trains. Major cities like Tokyo and Kyoto showcase vibrant urban life alongside historic temples and shrines. The country is famous for its cuisine, including sushi, ramen, and tempura. Natural beauty is abundant, from the iconic Mount Fuji to serene cherry blossom parks. Japan's unique blend of tradition and modernity, along with its warm hospitality, makes it a captivating destination for travelers.
Exiting...
